# 01 · Scenes and wind

Handoff §3 Steps 1 and 4, for **one site** chosen by the environment variable `DUNESIGHT_SITE` (default `seih_al_hama`; boxes live in `pipeline/sites.json`).

What this notebook does, in order:
1. **Access tests**: Earth Engine returns a Landsat scene over the box; earthaccess finds an HLS granule over the box.
2. **Landsat candidate table** (WRS-2 path/row from `sites.json`): Landsat 7 ETM+ before the SLC failure (31 May 2003), Landsat 8/9 OLI 2013-2025. Per scene: ID, date, sun angles, scene cloud, box coverage, in-box cloud / dilated cloud / cirrus / shadow from `QA_PIXEL`, a check that band 8 (15 m pan) is present and valid, a texture index and a haze indicator.
3. **One scene per year** for May and for November under the rule "reject if more than 1 % cloud/shadow inside the box", every rejection logged, B8 thumbnails of both sets, and a recommended month. Written to `pipeline/outputs/<site>/` as a **proposal awaiting confirmation**. Nothing full-resolution is downloaded here (that is notebook 02).
4. **Wind**: Fryberger drift potential from MERRA-2 hourly 10 m wind, 2000-2024, with unit tests of the maths, then a NASA POWER cross-check.
4. **Display-only frames 2003-2012** (section 3b): one November Landsat 7 scene per year after the scan-line-corrector failure, picked with the same in-box tests, for the app's timeline only. Written to a separate list, `pipeline/outputs/<site>/display_only_scenes_slc_off.json`; no measurement reads it.
5. Writes `public/data/<site>/wind_summary.json` (real values) and refreshes `public/data/sites.json`.

**One notebook for every site.** Everything site-specific comes from `pipeline/sites.json` (box, CRS, path/row, HLS tile, Phase 1 scene list) or from the per-site settings in this notebook (`SITE_QA`: visual-QA rejections and the zoom patch, each with what was seen). The written interpretation of each site's results lives in `pipeline/site_notes/<site>/` and is shown below next to tables generated from this run's numbers, so the numbers in a note can be checked against the run.

Run headless: `npm run pipeline` (all verified sites) or `DUNESIGHT_SITE=seih_al_hama .venv/bin/jupyter nbconvert --to notebook --execute 01_scenes_and_wind.ipynb --output-dir runs/seih_al_hama` from `pipeline/`. The executed copy with outputs is kept in `pipeline/runs/<site>/`; this source notebook stays without outputs.

Earth Engine results are cached in `pipeline/raw/<site>/` (gitignored). Set `DUNESIGHT_REFRESH=1` to re-query everything.

In [ ]:
# WHAT THIS DOES: finds the pipeline folder, imports the shared dunesight helpers, picks the site from
# DUNESIGHT_SITE, sets the output folders, and loads .env (only reports whether each secret is present).
import json, os, sys, time, math, io, datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

PIPELINE = next(p for p in [Path.cwd(), Path.cwd() / "pipeline", Path.cwd().parent, Path.cwd().parent.parent]
                if (p / "dunesight").is_dir())
sys.path.insert(0, str(PIPELINE))

import numpy as np
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import Markdown, display

from dunesight import config, site_index, wind, landsat, merra2

SITE = config.get_site()                      # raises if the site is unknown or not verified
SID = SITE["id"]
BBOX = config.bbox_list(SITE)                 # [W, S, E, N]
LAT_C, LON_C = config.centroid(SITE)
OUT = config.outputs_dir(SID); OUT.mkdir(parents=True, exist_ok=True)
RAW = config.raw_dir(SID); RAW.mkdir(parents=True, exist_ok=True)
PUB = config.public_dir(SID); PUB.mkdir(parents=True, exist_ok=True)
REFRESH = os.environ.get("DUNESIGHT_REFRESH", "") == "1"
load_dotenv(config.ROOT / ".env")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

print(f"site: {SID}  ({SITE['name']})  verified={SITE['verified']}")
print(f"bbox W,S,E,N: {BBOX}   centroid: {LAT_C:.4f} N, {LON_C:.4f} E   EPSG:{SITE['epsg']}   WRS-2 {SITE['wrs_path']}/{SITE['wrs_row']:03d}   HLS {SITE['hls_tile']}")
print("box source:", SITE["source"])
print("credentials present:", {k: bool(os.getenv(k)) for k in ("EE_PROJECT", "EARTHDATA_USERNAME", "EARTHDATA_PASSWORD")})
print("cache refresh forced:", REFRESH)
NOTES = PIPELINE / "site_notes" / SID          # written interpretation of this site's results (markdown)
REF_SITE = config.DEFAULT_SITE                 # the site whose proposed dates the others are compared with

## 1 · Access tests (handoff §5.2)

In [ ]:
# WHAT THIS DOES: initialises Earth Engine (project id from .env, not printed) and prints one Landsat 8
# Collection 2 Tier 1 scene over the box, as the §5.2 access test.
ee = landsat.ee_init(config.ROOT)
BOX = ee.Geometry.Rectangle(BBOX, "EPSG:4326", False)
first = (ee.ImageCollection("LANDSAT/LC08/C02/T1").filterBounds(BOX)
         .filter(ee.Filter.eq("WRS_PATH", SITE["wrs_path"])).filter(ee.Filter.eq("WRS_ROW", SITE["wrs_row"]))
         .filterDate("2024-05-01", "2024-06-01").first())
info = first.toDictionary(["LANDSAT_PRODUCT_ID", "DATE_ACQUIRED"]).getInfo()
print("Earth Engine OK. One Landsat scene over the box:", info["LANDSAT_PRODUCT_ID"], info["DATE_ACQUIRED"])

In [ ]:
# WHAT THIS DOES: logs in to NASA Earthdata with earthaccess (credentials from .env, never printed) and finds
# HLS L30 granules over the box, as the §5.2 access test (search only; nothing is downloaded). It also checks that
# the HLS tile named in sites.json is among them and that its granule footprint contains the whole box.
import logging, earthaccess
from shapely.geometry import Polygon, box as sbox
logging.getLogger("earthaccess").setLevel(logging.WARNING)
_auth = earthaccess.login(strategy="environment")
print("Earthdata login authenticated:", bool(_auth and _auth.authenticated))
gran = earthaccess.search_data(short_name="HLSL30", bounding_box=tuple(BBOX), temporal=("2024-05-01", "2024-05-31"), count=50)
tile_of = lambda g: g["meta"]["native-id"].split(".")[2]
mine = [g for g in gran if tile_of(g) == SITE["hls_tile"]]
print(f"HLSL30 granules over the box in May 2024: {len(gran)}, tiles {sorted({tile_of(g) for g in gran})}")
print(f"granules of the configured tile {SITE['hls_tile']} (first 5 shown):")
for g in mine[:5]:
    print("  ", g["meta"]["native-id"])
assert mine, f"configured HLS tile {SITE['hls_tile']} not found over the box"

def footprint(g):
    pts = g["umm"]["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]["GPolygons"][0]["Boundary"]["Points"]
    return Polygon([(p["Longitude"], p["Latitude"]) for p in pts])
_box = sbox(*BBOX)
cover = max(footprint(g).intersection(_box).area / _box.area for g in mine)
print(f"largest share of the box inside one {SITE['hls_tile']} granule footprint: {100 * cover:.1f} %")
assert cover > 0.999, "configured HLS tile does not contain the whole box"
del _auth


## 2 · Landsat candidate scenes

**Collections** (Earth Engine, Collection 2 Level-1 Tier 1): `LANDSAT/LE07/C02/T1` from 2000-01-01 up to the scan-line-corrector failure on **2003-05-31** (later L7 scenes have SLC-off gaps and are listed but excluded from the measurement; section 3b picks display-only timeline frames from them), and `LANDSAT/LC08/C02/T1` + `LANDSAT/LC09/C02/T1` for 2013-2025. Only the WRS-2 path/row from `sites.json` (printed in the first cell), so every scene shares one viewing geometry.

**In-box cloud** uses the `QA_PIXEL` bits over the valid (non-fill) 30 m pixels inside the box:
bit 0 fill, **bit 1 dilated cloud, bit 2 cirrus (OLI only; always 0 on ETM+), bit 3 cloud, bit 4 cloud shadow**. `pct_any_cloud_shadow` is the share of pixels with any of bits 1-4 set; it is the number the 1 % rule uses. Dilated cloud is a buffer around detected clouds, so including it is deliberately conservative. These are the USGS automatic (CFMask) flags: they miss thin dust haze, so a haze indicator is added below.

**Coverage**: the scene footprint contains the box *and* every 30 m QA pixel and 15 m B8 pixel in the box is valid (non-fill / non-zero).

**Band 8 check**: share of non-zero B8 pixels at 15 m, DN min/max/percentiles, and the share flagged saturated in `QA_RADSAT` bit 7 (band 8).

**Texture index** (for feature tracking): standard deviation of the B8 top-of-atmosphere reflectance after removing a 150 m-radius moving mean, divided by the mean reflectance. It measures 15 m-scale contrast (dune slip faces, shading). Higher is better for tracking. It is a relative index: ETM+ is 8-bit and OLI 12-bit (delivered as 16-bit), so compare it within a sensor.

**Haze indicator**: MERRA-2 hourly aerosol optical depth at 550 nm (total `TOTEXTTAU` and dust `DUEXTTAU`, `NASA/GSFC/MERRA/aer/2`) in the grid cell containing the box, for the hour of acquisition. It is a ~50 km model value, so an indicator of regional dust, not a measurement over the box. (The cell is chosen with the grid correction explained in section 4.)

In [ ]:
# WHAT THIS DOES: computes, on Earth Engine, the per-scene statistics described above for every path/row scene in
# the windows (one request per sensor-year, run in parallel), and caches the table in pipeline/raw/<site>/.
SCENE_CACHE = RAW / "landsat_scene_stats.csv"
EPSG = SITE["epsg"]
MERRA_TRANSFORM = ee.ImageCollection("NASA/GSFC/MERRA/slv/2").first().select("U10M").projection().getInfo()["transform"]
_cell = merra2.native_cell(LAT_C, LON_C)
AER_PT = ee.Geometry.Point(merra2.ee_pixel_centre(MERRA_TRANSFORM, _cell["i"], _cell["j"]))

def pr(cid):
    return (ee.ImageCollection(cid).filter(ee.Filter.eq("WRS_PATH", SITE["wrs_path"]))
            .filter(ee.Filter.eq("WRS_ROW", SITE["wrs_row"])))

WINDOWS = [("LANDSAT/LE07/C02/T1", y, f"{y}-01-01", f"{y + 1}-01-01" if y < 2003 else landsat.L7_SLC_FAILURE) for y in range(2000, 2004)]
WINDOWS += [(cid, y, f"{y}-01-01", f"{y + 1}-01-01") for cid in ("LANDSAT/LC08/C02/T1", "LANDSAT/LC09/C02/T1") for y in range(2013, 2026)]

def stats_for(w):
    cid, y, start, end = w
    fc = pr(cid).filterDate(start, end).map(lambda im: landsat.scene_stats(ee, im, BOX, EPSG, AER_PT, MERRA_TRANSFORM))
    for attempt in range(4):
        try:
            rows = [f["properties"] for f in fc.getInfo()["features"]]
            for r in rows:
                r["collection"] = cid
            return rows
        except Exception as e:  # transient EE errors: retry with back-off
            if attempt == 3:
                raise
            time.sleep(5 * (attempt + 1))

if SCENE_CACHE.exists() and not REFRESH:
    scenes = pd.read_csv(SCENE_CACHE)
    print(f"loaded cached scene statistics: {len(scenes)} scenes ({SCENE_CACHE.relative_to(config.ROOT)})")
else:
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        scenes = pd.DataFrame([r for rows in ex.map(stats_for, WINDOWS) for r in rows])
    scenes.to_csv(SCENE_CACHE, index=False)
    print(f"queried Earth Engine: {len(scenes)} scenes in {time.time() - t0:.0f} s")

scenes = landsat.derive(scenes)
scenes["date"] = pd.to_datetime(scenes["date"])
scenes["year"], scenes["month"] = scenes["date"].dt.year, scenes["date"].dt.month
scenes["sensor"] = scenes["collection"].map(landsat.COLLECTIONS)
scenes = scenes.sort_values("date").reset_index(drop=True)
print(scenes.groupby("sensor").agg(n=("scene_id", "size"), first=("date", "min"), last=("date", "max")))

In [ ]:
# WHAT THIS DOES: lists the Landsat 7 scenes on this path/row from the SLC failure (2003-05-31) onwards. They are
# excluded (scan-line gaps) and logged here and in the candidates CSV, without computing statistics for them.
slc = pr("LANDSAT/LE07/C02/T1").filterDate(landsat.L7_SLC_FAILURE, "2026-01-01")
slc_off = pd.DataFrame({"scene_id": slc.aggregate_array("LANDSAT_PRODUCT_ID").getInfo(),
                        "date": pd.to_datetime(slc.aggregate_array("DATE_ACQUIRED").getInfo())})
slc_off["collection"], slc_off["sensor"] = "LANDSAT/LE07/C02/T1", "Landsat 7 ETM+"
slc_off["excluded_reason"] = "Landsat 7 SLC-off (acquired on/after 2003-05-31)"
print(f"Excluded Landsat 7 SLC-off scenes on {SITE['wrs_path']}/{SITE['wrs_row']:03d}: {len(slc_off)} "
      f"({slc_off.date.min().date()} to {slc_off.date.max().date()})")
print("Last pre-SLC-failure L7 scene kept:", scenes[scenes.sensor == "Landsat 7 ETM+"].iloc[-1][["scene_id", "date"]].tolist())

In [ ]:
# WHAT THIS DOES: derives the pass/fail flags (box coverage, B8 validity, the 1 % cloud/shadow rule, the haze test) for every
# candidate, prints the B8 validity check, and summarises in-box cloud by sensor.
CLOUD_MAX_PCT = 1.0
scenes["covers_box"] = (scenes.footprint_contains_box == True) & (scenes.qa_valid_frac >= 0.999) & (scenes.b8_valid_frac >= 0.999)
scenes["b8_ok"] = (scenes.b8_valid_frac >= 0.999) & (scenes.b8_saturated_pct < 0.1) & (scenes.b8_p98 > scenes.b8_p2)
scenes["cloud_ok"] = scenes.pct_any_cloud_shadow <= CLOUD_MAX_PCT
# haze/dust test (QA_PIXEL does not flag dust): texture below half the median of the same calendar month's clear,
# covering scenes from the same sensor family
HAZE_FRACTION = 0.5
scenes["family"] = np.where(scenes.sensor == "Landsat 7 ETM+", "ETM+", "OLI")
_clear = scenes[scenes.covers_box & scenes.b8_ok & scenes.cloud_ok]
TEX_REF = _clear.groupby(["month", "family"]).b8_texture.median()
scenes["haze_ok"] = [t >= HAZE_FRACTION * TEX_REF.get((m, f), np.nan) for t, m, f in zip(scenes.b8_texture, scenes.month, scenes.family)]
print("median texture of clear covering scenes (month, family):")
print(TEX_REF.unstack().loc[[5, 11]].round(4))

print("Band 8 presence/validity over the box (all candidates):")
print(scenes.groupby("sensor").agg(n=("scene_id", "size"),
      b8_valid_frac_min=("b8_valid_frac", "min"), b8_sat_pct_max=("b8_saturated_pct", "max"),
      b8_dn_min=("b8_min", "min"), b8_dn_max=("b8_max", "max"), b8_ok=("b8_ok", "sum"), covers_box=("covers_box", "sum")))
bad = scenes[~scenes.b8_ok | ~scenes.covers_box]
print(f"\nScenes failing coverage or B8 checks: {len(bad)}")
if len(bad):
    print(bad[["scene_id", "date", "footprint_contains_box", "qa_valid_frac", "b8_valid_frac", "b8_saturated_pct"]].to_string(index=False))
print(f"\nScenes with in-box cloud/shadow <= {CLOUD_MAX_PCT} %: {int(scenes.cloud_ok.sum())} of {len(scenes)}")
print("In-box cloud/shadow % by sensor (any of bits 1-4):")
print(scenes.groupby("sensor").pct_any_cloud_shadow.describe(percentiles=[.5, .75, .9]).round(2))

In [ ]:
# WHAT THIS DOES: cross-checks the Earth Engine scene list against the Phase 1 USGS STAC list
# (the site's phase1_candidates_csv from sites.json, read only) and compares Earth Engine in-box cloud with STAC scene cloud.
stac_path = config.ROOT / SITE["phase1_candidates_csv"]
if stac_path.exists():
    stac = pd.read_csv(stac_path)
    stac = stac[(stac.wrs_path == SITE["wrs_path"]) & (stac.wrs_row == SITE["wrs_row"]) & (stac.collection_category == "T1")]
    ee_ids, stac_ids = set(scenes.scene_id), set(stac.scene_id)
    print(f"STAC T1 {SITE['wrs_path']}/{SITE['wrs_row']:03d} scenes: {len(stac_ids)}; Earth Engine: {len(ee_ids)}; in both: {len(ee_ids & stac_ids)}")
    print("only in Earth Engine:", sorted(ee_ids - stac_ids)[:20])
    print("only in STAC        :", sorted(stac_ids - ee_ids)[:20])
    both = scenes.merge(stac[["scene_id", "cloud_cover_scene"]].rename(columns={"cloud_cover_scene": "stac_cloud"}), on="scene_id")
    print("max |EE CLOUD_COVER - STAC eo:cloud_cover|:", round((both.cloud_cover_scene - both.stac_cloud).abs().max(), 3))
    lowscene_cloudy_box = scenes[(scenes.cloud_cover_scene <= 1) & ~scenes.cloud_ok]
    print(f"scenes with scene cloud <= 1 % but in-box cloud/shadow > 1 %: {len(lowscene_cloudy_box)}")
    print(lowscene_cloudy_box[["scene_id", "date", "cloud_cover_scene", "pct_any_cloud_shadow"]].assign(date=lowscene_cloudy_box.date.dt.date).round(2).to_string(index=False))
else:
    print(f"{SITE['phase1_candidates_csv']} not present; cross-check skipped")

In [ ]:
# WHAT THIS DOES: writes the full candidate table (every scene, plus the excluded SLC-off L7 scenes) to
# pipeline/outputs/<site>/landsat_candidates.csv.
cols = ["scene_id", "ee_id", "collection", "sensor", "date", "year", "month", "sun_elevation", "sun_azimuth",
        "cloud_cover_scene", "cloud_cover_land", "footprint_contains_box", "qa_valid_frac", "covers_box",
        "pct_cloud", "pct_dilated_cloud", "pct_cirrus", "pct_cloud_shadow", "pct_any_cloud_shadow", "cloud_ok", "haze_ok",
        "b8_valid_frac", "b8_min", "b8_max", "b8_p2", "b8_p98", "b8_saturated_pct", "b8_ok",
        "b8_texture", "family", "b8_toa_mean", "merra2_aod550_total", "merra2_aod550_dust"]
cand = pd.concat([scenes[cols].assign(excluded_reason=""), slc_off], ignore_index=True).sort_values("date")
cand["excluded_reason"] = cand["excluded_reason"].fillna("")
cand["date"] = cand["date"].dt.strftime("%Y-%m-%d")
cand.round(4).to_csv(OUT / "landsat_candidates.csv", index=False)
print(f"wrote {len(cand)} rows -> {(OUT / 'landsat_candidates.csv').relative_to(config.ROOT)}")

## 3 · One scene per year: May vs November

Rule, applied per calendar year to the scenes acquired in the month:
1. reject if the box is not fully covered or B8 fails the validity check;
2. reject if in-box cloud/shadow (any of QA bits 1-4) is **more than 1 %**;
3. reject as **haze/dust suspected** if the B8 texture index is below half the median texture of the month's other clear, covering scenes from the same sensor family (ETM+ or OLI). `QA_PIXEL` does not flag dust, and dust flattens the image;
4. reject scenes listed in this site's `VISUAL_REJECT` (from `SITE_QA` in the next cell): problems seen by eye in the thumbnails that neither QA nor the haze test caught. Each entry states what was seen. (Each site's list was filled after looking at that site's first-run thumbnails, and every pick was looked at; it is part of the method, not a hidden edit.);
5. among the survivors, pick the scene closest to a fixed target date (**20 May** or **10 November**) so the sun angle stays as constant as possible across years; ties go to the lower in-box cloud. Survivors not picked are logged as "passed, not picked" At every site other than the reference site (Seih Al Hama), a survivor acquired on one of the reference site's proposed dates is picked first, so the sites share acquisition dates wherever the data allow; the reference site uses the plain rule.

The 16 target years are 2000-2002 and 2013-2025 (handoff §3 Step 1). 2003 is reported as an optional extra: Landsat 7 still worked until 31 May 2003, so a May 2003 scene can exist; no November one can.

In [ ]:
# WHAT THIS DOES: applies the per-year selection rule for May and November and builds a rejection log with the
# reason for every scene not picked.
TARGET_YEARS = [2000, 2001, 2002] + list(range(2013, 2026))
TARGETS = {"May": (5, 20), "November": (11, 10)}
# Per-site settings found by looking at the images (not by the numbers). VISUAL_REJECT: scene -> what was seen.
# ZOOM: centre (lat, lon) and description of a ~3 x 3 km dune patch used for the zoom figure.
SITE_QA = {
    "seih_al_hama": {
        "VISUAL_REJECT": {
            "LE07_L1TP_160043_20020509_20200916_02_T1": (
                "visual QA: dark, irregular patches over roughly the SW quarter of the box in B8 and in true colour, "
                "also cooler in the thermal band (cloud shadow or rain-wetted sand); QA_PIXEL flags 0 % (ETM+ CFMask missed it)"),
        },
        "ZOOM": (24.058, 55.557, "dune belt E of the race track"),
        # display-only Landsat 7 SLC-off scenes (section 3b): problems seen by eye; never used for the measurement list
        "VISUAL_REJECT_DISPLAY": {},
    },
    "sweihan": {
        "VISUAL_REJECT": {
            "LC08_L1TP_160043_20181105_20200830_02_T1": (
                "visual QA: diffuse darker patch over the northern dune belt in B8, and cool patches in the thermal band over "
                "the N belt and a large part of the SW dune belt that are absent in other November scenes (thin cloud shadow or "
                "rain-wetted sand); QA_PIXEL flags 0 % here, while the same scene has 9.9 % cloud/cirrus over Seih Al Hama"),
        },
        "ZOOM": (24.495, 55.335, "northern dune belt NW of the military compound"),
        "VISUAL_REJECT_DISPLAY": {
            "LE07_L1TP_160043_20031104_20200915_02_T1": (
                "visual QA: diffuse dark patch over the W and SW of the box next to the farm belt in B8, and a broad cold swath across "
                "the centre and SW in the thermal band, absent from the other November scenes (cloud shadow or rain-wetted sand); "
                "QA_PIXEL flags 0.2 % (ETM+ CFMask missed most of it)"),
            "LE07_L1TP_160043_20081101_20200912_02_T1": (
                "visual QA: diffuse dark patches (cloud shadows) over the centre and W of the box in B8, cold over the E half in the "
                "thermal band; QA_PIXEL flags 0.2 %"),
        },
    },
    "liwa": {
        "VISUAL_REJECT": {
            "LE07_L1TP_160044_20020509_20200916_02_T1": (
                "visual QA: a dark plume about 1.5 km wide runs from the N edge into the NE quarter of the box in B8 and true "
                "colour, and is cold in the thermal band (cloud shadow or a rain/smoke streak); QA_PIXEL flags 0.001 % (ETM+ CFMask missed it)"),
            "LC08_L1TP_160044_20161115_20200905_02_T1": (
                "visual QA: thin, unflagged cloud or haze wisps (bluish-white in true colour, washing out the dune relief in B8) "
                "over roughly 1.5 x 1.5 km of the small-dune field N of the large farm block, centred near 23.048 N 54.149 E; QA_PIXEL flags 0 %"),
        },
        "ZOOM": (23.07, 54.15, "small-dune field in the NE of the box"),
        "VISUAL_REJECT_DISPLAY": {
            "LE07_L1TP_160044_20081101_20200912_02_T1": (
                "visual QA: bright, diffuse haze or thin cloud over the E third of the box in B8 (whitish in true colour), cold in the "
                "thermal band; QA_PIXEL flags 0.2 %"),
        },
    },
}
VISUAL_REJECT = SITE_QA[SID]["VISUAL_REJECT"]
print(f"visual-QA rejections for {SID}: {len(VISUAL_REJECT)}")

# Common dates across sites: at every site other than the reference, a scene acquired on a date in the reference
# site's proposed lists is preferred over a closer-to-target scene, as long as it passes every test here. The
# reference site itself is selected by the plain rule (no preference).
REF_DATES = set()
_ref_sel = config.outputs_dir(REF_SITE) / "scenes_selected.json"
if SID != REF_SITE and _ref_sel.exists():
    _r = json.loads(_ref_sel.read_text())
    REF_DATES = {pd.Timestamp(e["date"]) for k in ("recommended", "alternative") for e in _r[k]}
print(f"reference-site dates preferred: {len(REF_DATES)} ({'none: this is the reference site' if SID == REF_SITE else REF_SITE})")


def reject_reason(r, m):
    """Why a candidate scene fails the rule (None if it passes). m = calendar month (for the haze reference)."""
    if not r.covers_box:
        return "does not fully cover the box"
    if not r.b8_ok:
        return "B8 invalid/saturated in box"
    if not r.cloud_ok:
        return (f"in-box cloud/shadow {r.pct_any_cloud_shadow:.2f} % > {CLOUD_MAX_PCT} % "
                f"(cloud {r.pct_cloud:.2f}, dilated {r.pct_dilated_cloud:.2f}, cirrus {r.pct_cirrus:.2f}, shadow {r.pct_cloud_shadow:.2f})")
    if not r.haze_ok:
        return (f"haze/dust suspected: B8 texture {r.b8_texture:.3f} < {HAZE_FRACTION} x month median "
                f"{TEX_REF[(m, r.family)]:.3f} ({r.family}); MERRA-2 AOD550 {r.merra2_aod550_total:.2f}")
    if r.scene_id in VISUAL_REJECT:
        return VISUAL_REJECT[r.scene_id]
    return None

def select(month_name):
    m, d = TARGETS[month_name]
    picks, log = [], []
    for y in sorted(scenes.year.unique()):
        c = scenes[(scenes.year == y) & (scenes.month == m)].copy()
        if c.empty:
            log.append({"month": month_name, "year": y, "scene_id": None, "date": None, "reason": "no scene acquired on this path/row in this month"})
            continue
        c["days_from_target"] = (c.date - pd.Timestamp(y, m, d)).dt.days.abs()
        for _, r in c.iterrows():
            c.loc[_, "reject"] = reject_reason(r, m)
        c["not_ref_date"] = ~c.date.isin(REF_DATES)
        ok = c[c.reject.isna()].sort_values(["not_ref_date", "days_from_target", "pct_any_cloud_shadow"])
        if ok.empty:
            log.append({"month": month_name, "year": y, "scene_id": None, "date": None, "reason": "NO ACCEPTABLE SCENE this year"})
        else:
            picks.append(ok.iloc[0])
        for _, r in c.iterrows():
            if r.reject is not None and isinstance(r.reject, str):
                log.append({"month": month_name, "year": y, "scene_id": r.scene_id, "date": r.date.date(), "reason": r.reject})
            elif not ok.empty and r.scene_id != ok.iloc[0].scene_id:
                log.append({"month": month_name, "year": y, "scene_id": r.scene_id, "date": r.date.date(),
                            "reason": f"passed, not picked ({r.days_from_target} d from target vs {ok.iloc[0].days_from_target} d"
                                      + (f"; {REF_SITE} date preferred)" if not ok.iloc[0].not_ref_date and r.days_from_target < ok.iloc[0].days_from_target else ")")})
    return pd.DataFrame(picks).reset_index(drop=True), pd.DataFrame(log)

picks, logs = {}, {}
for mn in TARGETS:
    picks[mn], logs[mn] = select(mn)
rejections = pd.concat(logs.values(), ignore_index=True)
rejections.to_csv(OUT / "scene_rejections.csv", index=False)
print(f"rejection log: {len(rejections)} entries -> {(OUT / 'scene_rejections.csv').relative_to(config.ROOT)}")
for mn in TARGETS:
    print(f"\n--- {mn}: log ---")
    print(logs[mn].to_string(index=False))

In [ ]:
# WHAT THIS DOES: shows the per-year picks for both months side by side with in-box cloud, sun angles, texture
# and the MERRA-2 haze indicator, and lists which of the 16 target years each month covers.
show = ["year", "scene_id", "date", "pct_any_cloud_shadow", "pct_cloud", "pct_cloud_shadow", "cloud_cover_scene",
        "sun_elevation", "sun_azimuth", "b8_texture", "merra2_aod550_total", "merra2_aod550_dust"]
for mn in TARGETS:
    p = picks[mn]
    print(f"\n=== {mn} picks ({len(p)} years) ===")
    print(p[show].assign(date=p.date.dt.date).round(3).to_string(index=False))
    missing = [y for y in TARGET_YEARS if y not in set(p.year)]
    print(f"target years covered: {sum(y in set(p.year) for y in TARGET_YEARS)}/16; missing: {missing}; extra years: {sorted(set(p.year) - set(TARGET_YEARS))}")

# Off-month fill-in options for missing target years: the clean scene nearest the target date within +-35 days,
# outside the month. NOT added to the lists; offered for the user to accept or refuse.
FILL_INS = {}
ok_all = scenes[scenes.covers_box & scenes.b8_ok & scenes.cloud_ok & scenes.haze_ok & ~scenes.scene_id.isin(VISUAL_REJECT)]
for mn, (m, d) in TARGETS.items():
    FILL_INS[mn] = []
    for y in [y for y in TARGET_YEARS if y not in set(picks[mn].year)]:
        c = ok_all[(ok_all.year == y) & (ok_all.month != m)].copy()
        c["days"] = (c.date - pd.Timestamp(y, m, d)).dt.days.abs()
        c = c[c.days <= 35].sort_values("days")
        for _, r in c.iterrows():
            FILL_INS[mn].append({"year": y, "scene_id": r.scene_id, "date": r.date.strftime("%Y-%m-%d"), "days_from_target": int(r.days),
                                 "sun_elevation_deg": round(float(r.sun_elevation), 2), "sun_azimuth_deg": round(float(r.sun_azimuth), 2),
                                 "inbox_cloud_shadow_pct": round(float(r.pct_any_cloud_shadow), 3), "b8_texture": round(float(r.b8_texture), 4)})
        if c.empty:
            FILL_INS[mn].append({"year": y, "scene_id": None, "note": "no clean scene within 35 days of the target date"})
    print(f"\n{mn} off-month fill-in options (not used unless confirmed):")
    for f in FILL_INS[mn]:
        print("  ", f)

In [ ]:
# WHAT THIS DOES: fetches small B8 thumbnails (box only, ~28 m display pixels, stretched per scene from the in-box
# 2nd-98th percentile) of every pick from Earth Engine and plots May and November side by side for each year.
# Display only; no measurement uses these images.
def thumb(r):
    url = ee.Image(f"{r.collection}/{r.ee_id}").select("B8").getThumbURL(
        {"region": BOX, "dimensions": 360, "crs": f"EPSG:{EPSG}", "min": r.b8_p2, "max": r.b8_p98, "format": "png"})
    for attempt in range(4):
        try:
            resp = requests.get(url, timeout=120); resp.raise_for_status()
            return plt.imread(io.BytesIO(resp.content), format="png")
        except Exception:
            if attempt == 3:
                raise
            time.sleep(3)

all_years = sorted(set(picks["May"].year) | set(picks["November"].year))
jobs = [(mn, r) for mn in TARGETS for _, r in picks[mn].iterrows()]
with ThreadPoolExecutor(8) as ex:
    imgs = dict(zip([(mn, r.year) for mn, r in jobs], ex.map(lambda j: thumb(j[1]), jobs)))

ncol = 6
nrow = math.ceil(len(all_years) / (ncol // 2))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 2.6, nrow * 2.75))
for k, y in enumerate(all_years):
    for c, mn in enumerate(TARGETS):
        ax = axes.flat[2 * k + c]
        ax.set_xticks([]); ax.set_yticks([])
        if (mn, y) in imgs:
            r = picks[mn][picks[mn].year == y].iloc[0]
            ax.imshow(imgs[(mn, y)], cmap="gray")
            ax.set_title(f"{r.date.date()}  L{r.sensor.split()[1]}\nsun el {r.sun_elevation:.0f}°  tex {r.b8_texture:.3f}", fontsize=7.5)
        else:
            ax.set_facecolor("#dddddd"); ax.set_title(f"{y} {mn[:3]}: no scene", fontsize=7.5)
for ax in axes.flat[2 * len(all_years):]:
    ax.axis("off")
fig.suptitle(f"{SITE['name']}: Landsat B8 (15 m pan) picks, May (left) vs November (right) of each year\n"
             f"box W {BBOX[0]} S {BBOX[1]} E {BBOX[2]} N {BBOX[3]}, north up, display stretch per scene", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.965])
fig.savefig(OUT / "b8_thumbnails_may_vs_nov.png", dpi=110)
plt.show()

In [ ]:
# WHAT THIS DOES: zooms into the same ~3 x 3 km dune patch (centre chosen by eye from the thumbnails, SITE_QA["ZOOM"]) for one early and one recent year at full 15 m display resolution, so the difference in dune
# relief between May and November is visible.
ZOOM_LAT, ZOOM_LON, ZOOM_WHAT = SITE_QA[SID]["ZOOM"]
def crop(r, size_deg=0.0135):
    reg = ee.Geometry.Rectangle([ZOOM_LON - size_deg, ZOOM_LAT - size_deg, ZOOM_LON + size_deg, ZOOM_LAT + size_deg], "EPSG:4326", False)
    url = ee.Image(f"{r.collection}/{r.ee_id}").select("B8").getThumbURL(
        {"region": reg, "dimensions": 200, "crs": f"EPSG:{EPSG}", "min": r.b8_p2, "max": r.b8_p98, "format": "png"})
    return plt.imread(io.BytesIO(requests.get(url, timeout=120).content), format="png")

zoom_years = [y for y in (2001, 2024) if all(y in set(picks[mn].year) for mn in TARGETS)]
fig, axes = plt.subplots(1, 2 * len(zoom_years), figsize=(4.2 * len(zoom_years) * 2, 4.4))
for k, (y, mn) in enumerate([(y, mn) for y in zoom_years for mn in TARGETS]):
    r = picks[mn][picks[mn].year == y].iloc[0]
    axes[k].imshow(crop(r), cmap="gray"); axes[k].set_xticks([]); axes[k].set_yticks([])
    axes[k].set_title(f"{r.date.date()} ({r.sensor}), sun el {r.sun_elevation:.0f}° az {r.sun_azimuth:.0f}°", fontsize=8)
fig.suptitle(f"~3 x 3 km centred {ZOOM_LAT} N {ZOOM_LON} E ({ZOOM_WHAT}), 15 m pixels shown ~1:1", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "b8_zoom_may_vs_nov.png", dpi=110); plt.show()

In [ ]:
# WHAT THIS DOES: computes the comparison numbers behind the month choice: years covered, spread of sun elevation
# and azimuth across the picks, texture (all picks and per sensor), and haze (MERRA-2 AOD at acquisition) for the
# picks and for all candidate scenes in each month.
rows = []
for mn, (m, _) in TARGETS.items():
    p = picks[mn]
    allm = scenes[scenes.month == m]
    rows.append({
        "month": mn,
        "target_years_covered": f"{sum(y in set(p.year) for y in TARGET_YEARS)}/16",
        "sun_el_min": p.sun_elevation.min(), "sun_el_max": p.sun_elevation.max(), "sun_el_range": np.ptp(p.sun_elevation),
        "sun_az_range": np.ptp(p.sun_azimuth),
        "texture_median_all": p.b8_texture.median(),
        "texture_median_L7": p[p.sensor == "Landsat 7 ETM+"].b8_texture.median(),
        "texture_median_OLI": p[p.sensor != "Landsat 7 ETM+"].b8_texture.median(),
        "aod_total_median_picks": p.merra2_aod550_total.median(), "aod_total_max_picks": p.merra2_aod550_total.max(),
        "aod_dust_median_picks": p.merra2_aod550_dust.median(),
        "aod_total_median_all_scenes_in_month": allm.merra2_aod550_total.median(),
        "share_scenes_in_month_passing_1pct": allm.cloud_ok.mean(),
    })
comparison = pd.DataFrame(rows).set_index("month").T
print(comparison.round(3).to_string())

# Texture difference within the same year and sensor, November minus May (paired):
pair = picks["May"][["year", "b8_texture"]].merge(picks["November"][["year", "b8_texture"]], on="year", suffixes=("_may", "_nov"))
pair["nov_over_may"] = pair.b8_texture_nov / pair.b8_texture_may
print(f"\npaired years: {len(pair)}; November/May texture ratio median {pair.nov_over_may.median():.2f}, "
      f"min {pair.nov_over_may.min():.2f}, max {pair.nov_over_may.max():.2f}")

In [ ]:
# WHAT THIS DOES: prints, from this run's numbers, the facts the month recommendation rests on (years covered and
# what is missing, sun-angle spreads, texture, haze, rejections), so the written note below can be checked against them.
def fmt_years(p):
    return f"{sum(y in set(p.year) for y in TARGET_YEARS)}/16, missing {[y for y in TARGET_YEARS if y not in set(p.year)] or 'none'}"
lines = ["| | May (target 20 May) | November (target 10 Nov) |", "|---|---|---|"]
lines.append(f"| Target years with an accepted scene | {fmt_years(picks['May'])} | {fmt_years(picks['November'])} |")
for lab, key, f in [("Sun elevation min-max (deg)", None, None), ("Sun azimuth spread (deg)", "sun_az_range", "{:.1f}"),
                    ("Texture median, ETM+ picks", "texture_median_L7", "{:.3f}"), ("Texture median, OLI picks", "texture_median_OLI", "{:.3f}"),
                    ("MERRA-2 AOD550 at acquisition, picks: median / max", None, None),
                    ("Share of the month's scenes passing the 1 % rule", "share_scenes_in_month_passing_1pct", "{:.2f}")]:
    if lab.startswith("Sun elevation"):
        vals = [f"{comparison.loc['sun_el_min', mn]:.1f}-{comparison.loc['sun_el_max', mn]:.1f} (spread {comparison.loc['sun_el_range', mn]:.1f})" for mn in TARGETS]
    elif lab.startswith("MERRA-2"):
        vals = [f"{comparison.loc['aod_total_median_picks', mn]:.2f} / {comparison.loc['aod_total_max_picks', mn]:.2f} (dust median {comparison.loc['aod_dust_median_picks', mn]:.2f})" for mn in TARGETS]
    else:
        vals = [f.format(comparison.loc[key, mn]) for mn in TARGETS]
    lines.append(f"| {lab} | {vals[0]} | {vals[1]} |")
lines.append(f"| Same-year texture Nov / May | | median {pair.nov_over_may.median():.2f} (range {pair.nov_over_may.min():.2f}-{pair.nov_over_may.max():.2f}, "
             f"Nov higher in {int((pair.nov_over_may > 1).sum())} of {len(pair)} paired years) |")
for mn in TARGETS:
    rj = rejections[(rejections.month == mn) & rejections.reason.str.contains("haze|visual QA|cloud/shadow", regex=True, na=False)]
    lines.append(f"| {mn}: cloud, haze and visual rejections | " + ("; ".join(f"{r.date} ({r.reason.split(':')[0].split(' (')[0]})" for r in rj.itertuples()) or "none") + " | |")
display(Markdown(f"**Numbers from this run ({SID}, WRS-2 {SITE['wrs_path']}/{SITE['wrs_row']:03d}):**\n\n" + "\n".join(lines)))


In [ ]:
# WHAT THIS DOES: shows the written interpretation of the scene choice for this site (pipeline/site_notes/<site>/
# 01_scene_choice.md), written after looking at this site's numbers and thumbnails.
_f = NOTES / "01_scene_choice.md"
display(Markdown(_f.read_text()) if _f.exists() else Markdown(f"**No written note yet for {SID}** ({_f.relative_to(config.ROOT)})."))


### Same dates as the reference site

All three sites are on WRS-2 path 160 and are imaged in the same pass, so the plan is one calendar month and, as far as possible, the same acquisition dates everywhere. The next cell checks the reference site's proposed dates against this site's own in-box tests.

In [ ]:
# WHAT THIS DOES: compares this site with the reference site's proposed dates (all sites are on path 160, so the
# same dates exist). For each reference date it reports this site's in-box tests for the same-date scene and, where
# that scene fails or another one is picked, this site's substitute. Written into scenes_selected.json.
REF_SEL_PATH = config.outputs_dir(REF_SITE) / "scenes_selected.json"
COMMON = None
if SID == REF_SITE:
    print(f"{SID} is the reference site; nothing to compare.")
elif not REF_SEL_PATH.exists():
    print(f"reference list {REF_SEL_PATH.relative_to(config.ROOT)} not found; run {REF_SITE} first. Comparison skipped.")
else:
    ref = json.loads(REF_SEL_PATH.read_text())
    COMMON = {"reference_site": REF_SITE, "reference_status": ref["status"]}
    for key_ref, mn in [("recommended", ref["recommended_month"]), ("alternative", ref["alternative_month"])]:
        m = TARGETS[mn][0]
        ref_by_year = {e["year"]: e for e in ref[key_ref]}
        here = picks[mn].set_index("year")
        rows = []
        for y in TARGET_YEARS:
            e = ref_by_year.get(y)
            same = scenes[scenes.date == pd.Timestamp(e["date"])] if e else scenes.iloc[0:0]
            r = same.iloc[0] if len(same) else None
            pick = here.loc[y] if y in here.index else None
            rows.append({
                "year": y, "ref_date": e["date"] if e else None,
                "same_date_inbox_cloud_shadow_pct": None if r is None else round(float(r.pct_any_cloud_shadow), 3),
                "same_date_texture": None if r is None else round(float(r.b8_texture), 4),
                "same_date_aod550": None if r is None else round(float(r.merra2_aod550_total), 3),
                "same_date_result": ("no reference date" if e is None else "scene not on this path/row" if r is None
                                     else (reject_reason(r, m) or "passes")),
                "pick_here": None if pick is None else pick.date.strftime("%Y-%m-%d"),
                "pick_inbox_cloud_shadow_pct": None if pick is None else round(float(pick.pct_any_cloud_shadow), 3),
                "same_as_reference": bool(e is not None and pick is not None and pick.date.strftime("%Y-%m-%d") == e["date"]),
            })
        t = pd.DataFrame(rows)
        COMMON[mn] = {"n_same_dates": int(t.same_as_reference.sum()), "n_reference_dates": int(t.ref_date.notna().sum()),
                      "differences": t[~t.same_as_reference].to_dict("records"), "all": t.to_dict("records")}
        print(f"\n=== {mn}: {SID} vs {REF_SITE} ({key_ref} list of the reference) ===")
        print(t.to_string(index=False))
        print(f"same date as the reference in {COMMON[mn]['n_same_dates']} of {COMMON[mn]['n_reference_dates']} reference years; "
              f"years that differ: {t[~t.same_as_reference].year.tolist()}")


In [ ]:
# WHAT THIS DOES: writes pipeline/outputs/<site>/scenes_selected.json with the recommended month's list and the
# alternative month's list, marked as a proposal awaiting the user's confirmation.
RECOMMENDED, ALTERNATIVE = "November", "May"

def as_list(p):
    out = []
    for _, r in p.iterrows():
        out.append({"year": int(r.year), "date": r.date.strftime("%Y-%m-%d"), "scene_id": r.scene_id,
                    "ee_asset": f"{r.collection}/{r.ee_id}", "sensor": f"{r.sensor} B8", "pixel_m": 15,
                    "sun_elevation_deg": round(float(r.sun_elevation), 2), "sun_azimuth_deg": round(float(r.sun_azimuth), 2),
                    "cloud_cover_scene_pct": round(float(r.cloud_cover_scene), 2),
                    "inbox_cloud_shadow_pct": round(float(r.pct_any_cloud_shadow), 3),
                    "b8_texture": round(float(r.b8_texture), 4),
                    "merra2_aod550_total": round(float(r.merra2_aod550_total), 3),
                    "in_16_target_years": int(r.year) in TARGET_YEARS})
    return out

sel = {
    "status": "proposed, awaiting user confirmation",
    "site": SID, "site_name": SITE["name"], "bbox": SITE["bbox"],
    "wrs_path": SITE["wrs_path"], "wrs_row": SITE["wrs_row"],
    "generated_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "rule": (f"Per calendar year, scenes in the month on WRS-2 {SITE['wrs_path']}/{SITE['wrs_row']:03d} (Collection 2 Tier 1) "
             f"that fully cover the box with valid B8; reject if in-box cloud/dilated cloud/cirrus/shadow "
             f"(QA_PIXEL bits 1-4) > {CLOUD_MAX_PCT} %; pick the survivor closest to the target date "
             "(20 May / 10 Nov). Landsat 7 only before 2003-05-31 (SLC failure)."),
    "visual_reject": VISUAL_REJECT,
    "haze_rule": f"reject if B8 texture < {HAZE_FRACTION} x median texture of the month's clear covering scenes of the same sensor family",
    "recommended_month": RECOMMENDED,
    "recommended": as_list(picks[RECOMMENDED]),
    "alternative_month": ALTERNATIVE,
    "alternative": as_list(picks[ALTERNATIVE]),
    "off_month_fill_in_options": FILL_INS,
    "comparison": {k: {kk: (round(float(vv), 4) if isinstance(vv, (float, np.floating)) else vv) for kk, vv in v.items()}
                   for k, v in comparison.to_dict().items()},
    "files": {"candidates": "landsat_candidates.csv", "rejections": "scene_rejections.csv",
              "thumbnails": "b8_thumbnails_may_vs_nov.png", "zoom": "b8_zoom_may_vs_nov.png"},
    "note": "Nothing has been downloaded at full resolution. Notebook 02 uses the confirmed list.",
}
if COMMON is not None:
    sel["same_dates_as_reference_site"] = COMMON
(OUT / "scenes_selected.json").write_text(json.dumps(sel, indent=1) + "\n")
print(f"wrote {(OUT / 'scenes_selected.json').relative_to(config.ROOT)}: {RECOMMENDED} {len(sel['recommended'])} scenes (recommended), "
      f"{ALTERNATIVE} {len(sel['alternative'])} scenes (alternative)")
for s in sel["recommended"]:
    print(f"  {s['year']}  {s['date']}  {s['scene_id']}")

## 3b · Display-only Landsat 7 frames, 2003-2012 (scan-line gaps)

**Decision (Sultan, 2026-11-15).** The timeline had nothing between 2001 and 2013. There is no November 2002 scene on these paths, and on **31 May 2003** Landsat 7's scan line corrector failed, so every later ETM+ image has wedge-shaped stripes of missing data that widen away from the centre of the swath. These images are now used **for display only**: one frame per year where an acceptable November scene exists, with the stripes left as no-data (nothing is interpolated). They do not enter the tracking network, the Landsat 7 long-baseline check, the registration, any statistic, the threat calculation or the projection. They are kept in their own list, `pipeline/outputs/<site>/display_only_scenes_slc_off.json`, which notebook 02 never reads; the confirmed measurement list stays `pipeline/site_config/<site>/scenes_confirmed.json`.

**Rule.** The tests of section 3, with two changes forced by the stripes: (1) *coverage* means the box lies inside the scene footprint; the stripes themselves are allowed (their share of the box is reported per scene); (2) *in-box cloud/shadow* (QA bits 1-4, ≤ 1 %) and the B8 checks are computed over the pixels that have data. The haze test uses the same reference as section 3 (half the median texture of the clear, covering November ETM+ scenes, i.e. 2000 and 2001). Every candidate is looked at in B8, true colour and the thermal band (ETM+ cloud flags have missed cloud shadow here before); problems seen by eye go in `SITE_QA[site]["VISUAL_REJECT_DISPLAY"]` with what was seen. November only (no off-month fill-ins, as decided for the measurement list); among the survivors the scene closest to 10 November is picked. Tier 1 only, like the measurement list; Tier 2 scenes are counted in the log. A year with no acceptable scene stays empty, with its reason in `scene_rejections.csv` (rows with `list` = `display_only`).

In [ ]:
# WHAT THIS DOES: computes the same per-scene in-box statistics as section 2 for the Landsat 7 November scenes acquired
# after the scan-line-corrector failure, 2003-2012 (Tier 1; cached in pipeline/raw/<site>/), and lists any Tier 2 scenes.
DISPLAY_YEARS = list(range(2003, 2013))
SLC_CACHE = RAW / "landsat_scene_stats_slc_off_nov.csv"

def nov_slc_off(cid):
    return (pr(cid).filterDate(landsat.L7_SLC_FAILURE, f"{DISPLAY_YEARS[-1] + 1}-01-01")
            .filter(ee.Filter.calendarRange(11, 11, "month")))

if SLC_CACHE.exists() and not REFRESH:
    slc_nov = pd.read_csv(SLC_CACHE)
    print(f"loaded cached SLC-off November statistics: {len(slc_nov)} scenes ({SLC_CACHE.relative_to(config.ROOT)})")
else:
    for attempt in range(4):
        try:
            fc = nov_slc_off("LANDSAT/LE07/C02/T1").map(lambda im: landsat.scene_stats(ee, im, BOX, EPSG, AER_PT, MERRA_TRANSFORM))
            slc_nov = pd.DataFrame([f["properties"] for f in fc.getInfo()["features"]])
            break
        except Exception:
            if attempt == 3:
                raise
            time.sleep(5 * (attempt + 1))
    slc_nov["collection"] = "LANDSAT/LE07/C02/T1"
    slc_nov.to_csv(SLC_CACHE, index=False)
    print(f"queried Earth Engine: {len(slc_nov)} SLC-off November scenes")
slc_nov = landsat.derive(slc_nov)
slc_nov["date"] = pd.to_datetime(slc_nov["date"])
slc_nov["year"], slc_nov["month"] = slc_nov["date"].dt.year, slc_nov["date"].dt.month
slc_nov["sensor"], slc_nov["family"] = "Landsat 7 ETM+", "ETM+"
slc_nov["gap_pct"] = 100 * (1 - slc_nov["b8_valid_frac"])     # 15 m B8 pixels in the box with no data (scan-line gaps)
slc_nov = slc_nov.sort_values("date").reset_index(drop=True)
T2_NOV = nov_slc_off("LANDSAT/LE07/C02/T2").aggregate_array("LANDSAT_PRODUCT_ID").getInfo()
print(f"Tier 2 SLC-off November scenes on this path/row: {len(T2_NOV)} {T2_NOV}")
print(slc_nov[["scene_id", "date", "sun_elevation", "cloud_cover_scene", "footprint_contains_box", "gap_pct", "pct_any_cloud_shadow",
               "pct_cloud", "pct_cloud_shadow", "b8_saturated_pct", "b8_texture", "merra2_aod550_total"]]
      .assign(date=slc_nov.date.dt.date).round(3).to_string(index=False))

In [ ]:
# WHAT THIS DOES: shows every candidate in B8 (15 m pan), true colour (B3/B2/B1) and the thermal band (B6, low gain), box
# only, so cloud shadow or wet sand that QA_PIXEL missed can be seen; saved to pipeline/outputs/<site>/. Display only.
def thumb_bands(r, bands, vis, dim=360):
    url = ee.Image(f"{r.collection}/{r.ee_id}").select(bands).getThumbURL(
        {"region": BOX, "dimensions": dim, "crs": f"EPSG:{EPSG}", "format": "png", **vis})
    for attempt in range(4):
        try:
            resp = requests.get(url, timeout=120); resp.raise_for_status()
            return plt.imread(io.BytesIO(resp.content), format="png")
        except Exception:
            if attempt == 3:
                raise
            time.sleep(3)

def views(r):
    t = ee.Image(f"{r.collection}/{r.ee_id}").select("B6_VCID_1").reduceRegion(
        ee.Reducer.percentile([2, 98]), BOX, 60, f"EPSG:{EPSG}").getInfo()
    return (thumb_bands(r, "B8", {"min": r.b8_p2, "max": r.b8_p98}),
            thumb_bands(r, ["B3", "B2", "B1"], {"min": 40, "max": 255}),
            thumb_bands(r, "B6_VCID_1", {"min": t["B6_VCID_1_p2"], "max": t["B6_VCID_1_p98"], "palette": ["000080", "00ffff", "ffff00", "ff0000"]}))

with ThreadPoolExecutor(8) as ex:
    VIEWS = list(ex.map(views, [r for _, r in slc_nov.iterrows()]))
fig, axes = plt.subplots(len(slc_nov), 3, figsize=(10.5, 3.6 * len(slc_nov)), squeeze=False)
for k, ((_, r), ims) in enumerate(zip(slc_nov.iterrows(), VIEWS)):
    for ax, im, nm in zip(axes[k], ims, ("B8 pan", "true colour", "thermal B6")):
        ax.imshow(im, cmap="gray"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{r.date.date()} {nm}" + (f"\ncloud/shadow {r.pct_any_cloud_shadow:.2f} %, gaps {r.gap_pct:.0f} %" if nm == "B8 pan" else ""), fontsize=7.5)
fig.suptitle(f"{SITE['name']}: Landsat 7 SLC-off November candidates 2003-2012 (white = scan-line gaps), display stretch per scene", fontsize=9)
fig.tight_layout(rect=[0, 0, 1, 0.985])
fig.savefig(OUT / "l7_slc_off_nov_visual_check.png", dpi=80)
plt.show()

In [ ]:
# WHAT THIS DOES: applies the rule to those scenes, picks at most one per year (closest to 10 November), logs every
# rejection and every empty year in scene_rejections.csv with list = "display_only", and writes the separate list
# pipeline/outputs/<site>/display_only_scenes_slc_off.json. Nothing here enters the measurement.
VISUAL_REJECT_DISPLAY = SITE_QA[SID].get("VISUAL_REJECT_DISPLAY", {})
assert not set(VISUAL_REJECT_DISPLAY) & set(VISUAL_REJECT), "a scene may be in only one visual-QA list"
TEX_REF_ETM_NOV = float(TEX_REF[(11, "ETM+")])          # the section 3 reference: clear, covering November ETM+ scenes (2000-2001)

def display_reject_reason(r):
    if not r.footprint_contains_box:
        return "does not fully cover the box (scene footprint)"
    if not ((r.b8_saturated_pct < 0.1) and (r.b8_p98 > r.b8_p2)):
        return "B8 saturated or without contrast in the box"
    if r.pct_any_cloud_shadow > CLOUD_MAX_PCT:
        return (f"in-box cloud/shadow {r.pct_any_cloud_shadow:.2f} % > {CLOUD_MAX_PCT} % of the pixels with data "
                f"(cloud {r.pct_cloud:.2f}, dilated {r.pct_dilated_cloud:.2f}, shadow {r.pct_cloud_shadow:.2f})")
    if r.b8_texture < HAZE_FRACTION * TEX_REF_ETM_NOV:
        return (f"haze/dust suspected: B8 texture {r.b8_texture:.3f} < {HAZE_FRACTION} x November ETM+ median {TEX_REF_ETM_NOV:.3f}; "
                f"MERRA-2 AOD550 {r.merra2_aod550_total:.2f}")
    if r.scene_id in VISUAL_REJECT_DISPLAY:
        return VISUAL_REJECT_DISPLAY[r.scene_id]
    return None

m_, d_ = TARGETS["November"]
disp_picks, disp_log, empty_years = [], [], []
for y in DISPLAY_YEARS:
    c = slc_nov[slc_nov.year == y].copy()
    t2 = [s for s in T2_NOV if s.split("_")[3][:4] == str(y)]
    if c.empty:
        why = ("no Landsat 7 scene acquired on this path/row in November (Collection 2 Tier 1"
               + (f"; Tier 2 only: {', '.join(t2)})" if t2 else " or Tier 2)"))
        disp_log.append({"month": "November", "year": y, "scene_id": None, "date": None, "reason": why, "list": "display_only"})
        empty_years.append({"year": y, "reason": why})
        continue
    c["days_from_target"] = (c.date - pd.Timestamp(y, m_, d_)).dt.days.abs()
    c["reject"] = [display_reject_reason(r) for _, r in c.iterrows()]
    ok = c[c.reject.isna()].sort_values(["days_from_target", "pct_any_cloud_shadow"])
    if ok.empty:
        why = "no acceptable November scene: " + "; ".join(f"{r.date.date()} {r.reject.split(':')[0].split(' (')[0]}" for _, r in c.iterrows())
        disp_log.append({"month": "November", "year": y, "scene_id": None, "date": None, "reason": "NO ACCEPTABLE SCENE this year", "list": "display_only"})
        empty_years.append({"year": y, "reason": why})
    else:
        disp_picks.append(ok.iloc[0])
    for _, r in c.iterrows():
        if isinstance(r.reject, str):
            disp_log.append({"month": "November", "year": y, "scene_id": r.scene_id, "date": r.date.date(), "reason": r.reject, "list": "display_only"})
        elif not ok.empty and r.scene_id != ok.iloc[0].scene_id:
            disp_log.append({"month": "November", "year": y, "scene_id": r.scene_id, "date": r.date.date(), "list": "display_only",
                             "reason": f"passed, not picked ({r.days_from_target} d from target vs {ok.iloc[0].days_from_target} d)"})
disp_picks = pd.DataFrame(disp_picks).reset_index(drop=True)
disp_log = pd.DataFrame(disp_log)

# one rejection log for both lists: the measurement rows from section 3 keep list = "measurement"
rej_all = pd.read_csv(OUT / "scene_rejections.csv")
if "list" not in rej_all:
    rej_all["list"] = "measurement"
rej_all = pd.concat([rej_all[rej_all.list != "display_only"], disp_log], ignore_index=True)
rej_all.to_csv(OUT / "scene_rejections.csv", index=False)
print(f"rejection log: {len(disp_log)} display-only entries added -> {(OUT / 'scene_rejections.csv').relative_to(config.ROOT)}")
print(disp_log.to_string(index=False))

_conf_p = PIPELINE / "site_config" / SID / "scenes_confirmed.json"
conf_ids = {e["scene_id"] for e in json.loads(_conf_p.read_text())["scenes"]} if _conf_p.exists() else set()
assert not set(disp_picks.get("scene_id", [])) & conf_ids, "a display-only scene is in the confirmed measurement list"
nov_log = logs["November"]
before = [{"year": int(y), "reason": nov_log[(nov_log.year == y) & nov_log.scene_id.isna()].reason.iloc[0]}
          for y in range(2002, DISPLAY_YEARS[0]) if ((nov_log.year == y) & nov_log.scene_id.isna()).any()]
disp = {
    "status": "display only: not used for any measurement",
    "not_used_in": ["tracking network (velocities)", "Landsat 7 long-baseline check", "registration / stable-ground offsets",
                    "any statistic", "threat (reach years)", "projection"],
    "decision": "Sultan, 2026-11-15: fill the 2003-2012 timeline gap with the striped Landsat 7 images for display only, stripes shown as missing data.",
    "site": SID, "site_name": SITE["name"], "bbox": SITE["bbox"], "wrs_path": SITE["wrs_path"], "wrs_row": SITE["wrs_row"],
    "generated_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "slc_note": ("Landsat 7's scan line corrector failed on 31 May 2003. Every later ETM+ image has wedge-shaped stripes with no data "
                 "(about 22 % of a full scene), narrow near the centre line of the swath and widening toward its edges."),
    "rule": (f"November Landsat 7 ETM+ scenes (Collection 2 Tier 1) on WRS-2 {SITE['wrs_path']}/{SITE['wrs_row']:03d}, {DISPLAY_YEARS[0]}-{DISPLAY_YEARS[-1]}; "
             "the box inside the scene footprint (stripes allowed); B8 not saturated; in-box cloud/dilated cloud/shadow (QA_PIXEL bits 1-4) "
             f"<= {CLOUD_MAX_PCT} % of the pixels with data; B8 texture >= {HAZE_FRACTION} x the median of the clear covering November ETM+ "
             f"scenes ({TEX_REF_ETM_NOV:.4f}); not rejected by eye (visual_reject); closest to 10 November."),
    "visual_reject": VISUAL_REJECT_DISPLAY,
    "scenes": [{"year": int(r.year), "date": r.date.strftime("%Y-%m-%d"), "scene_id": r.scene_id, "ee_asset": f"{r.collection}/{r.ee_id}",
                "sensor": "Landsat 7 ETM+ B8", "pixel_m": 15, "sun_elevation_deg": round(float(r.sun_elevation), 2),
                "sun_azimuth_deg": round(float(r.sun_azimuth), 2), "cloud_cover_scene_pct": round(float(r.cloud_cover_scene), 2),
                "inbox_cloud_shadow_pct_of_data": round(float(r.pct_any_cloud_shadow), 3), "inbox_b8_gap_pct": round(float(r.gap_pct), 1),
                "b8_texture": round(float(r.b8_texture), 4), "merra2_aod550_total": round(float(r.merra2_aod550_total), 3),
                "use": "display_only"} for _, r in disp_picks.iterrows()],
    "empty_years": empty_years,
    "empty_years_before": before,
    "files": {"rejections": "scene_rejections.csv (list = display_only)", "visual_check": "l7_slc_off_nov_visual_check.png"},
}
(OUT / "display_only_scenes_slc_off.json").write_text(json.dumps(disp, indent=1) + "\n")
print(f"\nwrote {(OUT / 'display_only_scenes_slc_off.json').relative_to(config.ROOT)}: {len(disp['scenes'])} display-only scenes; "
      f"empty years {[e['year'] for e in empty_years]}")
for s in disp["scenes"]:
    print(f"  {s['year']}  {s['date']}  {s['scene_id']}  gaps {s['inbox_b8_gap_pct']} % of the box, cloud/shadow {s['inbox_cloud_shadow_pct_of_data']} %")

## 4 · Wind: Fryberger drift potential from MERRA-2

**Method** (Fryberger & Dean 1979, "Dune forms and wind regime", USGS Prof. Paper 1052, ch. F):
- V = hourly-mean wind speed at 10 m in **knots** (`U10M`, `V10M` in m/s × 1/0.514444); threshold **Vt = 12 kt**.
- **DP = Σ V²(V − Vt)/100 · t**, summed over hours with V > Vt, where **t is the percentage of the period that hour represents**. With hourly data each hour is its own class, so for a year t = 100/N (N = valid hours in that year), and DP_year = mean over all hours of V²(V − Vt)₊. DP is in vector units (VU) per year, which is what the class limits refer to: **low < 200 ≤ intermediate < 400 ≤ high**. (The handoff's shorthand "DP = U²(U − Ut)" leaves out the /100 · t normalisation; without it the numbers scale with record length and the class limits mean nothing.)
- **FROM direction** (meteorological) = atan2(−U, −V); sand moves **downwind** = FROM + 180°.
- Each hour contributes a vector of length V²(V − Vt)/100 · t pointing downwind. **RDP** = length of the vector sum, **RDD** = its bearing (direction sand moves *to*), **RDP/DP** = directional variability (near 1 = one wind direction; small = opposing winds).
- Whole period: DP = mean annual DP; RDP/RDD from the mean of the yearly resultant vectors (each year weighted equally); sand rose = mean annual DP per 16 sectors (22.5°, sector centred on its bearing), binned by **FROM** direction, the usual Fryberger sand-rose convention.
- Missing hours are dropped and t uses the valid count (the valid hours represent the year); the count is reported.

The maths lives in `pipeline/dunesight/wind.py`; the next cell tests it on cases that can be checked by hand.

In [ ]:
# WHAT THIS DOES: unit-tests the direction and DP maths on synthetic winds whose answers can be worked out by hand.
KT = 0.514444  # m/s per knot
def ms(kt): return kt * KT
def close(a, b, tol=1e-6): return abs(a - b) <= tol * max(1.0, abs(b))
N = 8760
tests = []

# 1. A pure northerly (blowing FROM north, so U = 0, V < 0): FROM 0, downwind 180.
tests.append(("northerly FROM = 0", close(float(wind.from_dir_deg(0.0, -5.0)), 0.0)))
tests.append(("northerly downwind = 180", close(float(wind.downwind_deg(0.0, -5.0)), 180.0)))
# 2. Easterly (U < 0): FROM 90, downwind 270. Westerly (U > 0): FROM 270.
tests.append(("easterly FROM = 90", close(float(wind.from_dir_deg(-5.0, 0.0)), 90.0)))
tests.append(("westerly FROM = 270", close(float(wind.from_dir_deg(5.0, 0.0)), 270.0)))
# 3. A NW shamal (blowing toward SE: U > 0, V < 0): FROM 315, downwind 135.
tests.append(("NW wind FROM = 315", close(float(wind.from_dir_deg(1.0, -1.0)), 315.0)))
tests.append(("NW wind downwind = 135", close(float(wind.downwind_deg(1.0, -1.0)), 135.0)))
# 4. 20 kt = 10.28888 m/s converts back to 20 kt.
tests.append(("20 kt round trip", close(float(wind.speed_kt(ms(20), 0)), 20.0)))
# 5. Constant 20 kt northerly all year: DP = 20^2 * (20-12) / 100 * 100 = 3200 VU, RDP = DP, RDD = 180, RDP/DP = 1.
r = wind.fryberger(np.zeros(N), np.full(N, -ms(20)))
tests.append(("constant 20 kt: DP = 3200", close(r["DP"], 3200.0)))
tests.append(("constant 20 kt: RDP = 3200, RDD = 180", close(r["RDP"], 3200.0) and close(r["RDD_deg"], 180.0)))
tests.append(("constant 20 kt: rose all in FROM sector 0", close(r["rose_from"][0], 3200.0) and close(r["rose_from"][1:].sum(), 0.0)))
# 6. Below threshold (10 kt) and exactly at threshold (12 kt): DP = 0.
tests.append(("10 kt: DP = 0", wind.fryberger(np.zeros(N), np.full(N, ms(10)))["DP"] == 0.0))
tests.append(("12 kt: DP = 0", close(wind.fryberger(np.zeros(N), np.full(N, ms(12)))["DP"], 0.0, 1e-9)))
# 7. 25 % of hours a 20 kt westerly, rest calm: DP = 3200 * 0.25 = 800, sand moves east (RDD 90).
u = np.zeros(N); u[: N // 4] = ms(20)
r = wind.fryberger(u, np.zeros(N))
tests.append(("25 % westerly 20 kt: DP = 800, RDD = 90", close(r["DP"], 800.0) and close(r["RDD_deg"], 90.0)))
# 8. Half the year from N, half from S at 20 kt: DP = 3200, RDP = 0 (opposing winds cancel).
v = np.where(np.arange(N) < N // 2, -ms(20), ms(20))
r = wind.fryberger(np.zeros(N), v)
tests.append(("opposing N/S: DP = 3200, RDP = 0", close(r["DP"], 3200.0) and r["RDP"] < 1e-6))
# 9. Half from N, half from E at 20 kt: each 1600 VU; resultant 1600*sqrt(2) = 2262.74 toward 225 (SW).
u = np.where(np.arange(N) < N // 2, 0.0, -ms(20)); v = np.where(np.arange(N) < N // 2, -ms(20), 0.0)
r = wind.fryberger(u, v)
tests.append(("N + E: RDP = 2262.74, RDD = 225", close(r["RDP"], 1600 * math.sqrt(2)) and close(r["RDD_deg"], 225.0)))
# 10. Missing hours (NaN) are dropped and t uses the valid count: same DP as test 5.
v = np.full(N, -ms(20)); v[:100] = np.nan
tests.append(("missing hours ignored", close(wind.fryberger(np.zeros(N), v)["DP"], 3200.0) and wind.fryberger(np.zeros(N), v)["n_valid"] == N - 100))
# 11. Sector binning: FROM 350 and 11.2 -> sector 0; 11.3 -> sector 1 (22.5); 348.7 -> sector 15 (337.5).
tests.append(("sector binning", list(wind.sector_index([350, 11.2, 11.3, 348.7])) == [0, 0, 1, 15]))
# 12. DP class limits.
tests.append(("DP classes", [wind.dp_class(x) for x in (199.9, 200, 399.9, 400)] == ["low", "intermediate", "intermediate", "high"]))
# 13. One hour at 30 kt in a 100-hour record: DP = 900*18/100 * (100/100) = 162.
v = np.zeros(100); v[0] = -ms(30)
tests.append(("one 30 kt hour in 100: DP = 162", close(wind.fryberger(np.zeros(100), v)["DP"], 162.0)))

for name, ok in tests:
    print(("PASS " if ok else "FAIL ") + name)
assert all(ok for _, ok in tests), "wind maths unit test failed"
print(f"all {len(tests)} synthetic tests passed")

### Which MERRA-2 grid cell?

MERRA-2's native grid has cell centres at latitude −90 + 0.5 j and longitude −180 + 0.625 i. Earth Engine's `NASA/GSFC/MERRA/slv/2` declares a pixel grid whose origin is (−180.5, −90.5), which puts every pixel **0.1875° west and 0.25° south** of the native cell with the same index. The next cell checks this against NASA POWER (which serves the native MERRA-2 cells) for one hour, and then samples the Earth Engine pixel that holds the native cell containing the box centroid, instead of the pixel Earth Engine would pick at the centroid's coordinates.

In [ ]:
# WHAT THIS DOES: reads Earth Engine's MERRA-2 pixel grid, works out the native cell containing the box centroid,
# and verifies for one hour which Earth Engine pixel holds that cell by comparing with NASA POWER at the centroid.
slv = ee.ImageCollection("NASA/GSFC/MERRA/slv/2")
proj = slv.first().select("U10M").projection().getInfo()
T = proj["transform"]
cell = merra2.native_cell(LAT_C, LON_C)
naive_ij = merra2.ee_pixel_at(T, LAT_C, LON_C)
print("Earth Engine MERRA-2 projection:", proj["crs"], T)
print(f"native MERRA-2 cell containing the centroid: i={cell['i']} j={cell['j']}, centre {cell['lat_c']} N {cell['lon_c']} E, "
      f"spans {cell['south']}-{cell['north']} N, {cell['west']}-{cell['east']} E (0.5 deg x 0.625 deg, about "
      f"{0.5 * 110.6:.0f} km x {0.625 * 111.32 * math.cos(math.radians(cell['lat_c'])):.0f} km)")
print(f"pixel Earth Engine would sample at the centroid coordinates: i={naive_ij[0]} j={naive_ij[1]}")
assert cell["south"] <= LAT_C < cell["north"] and cell["west"] <= LON_C < cell["east"], "native cell does not contain the centroid"
print(f"check: centroid {LAT_C:.4f} N {LON_C:.4f} E lies inside the cell; distance to the nearest cell edge "
      f"{min(LAT_C - cell['south'], cell['north'] - LAT_C, (LON_C - cell['west']) * math.cos(math.radians(LAT_C)), (cell['east'] - LON_C) * math.cos(math.radians(LAT_C))) * 111.2:.1f} km")
# which native cells does the box overlap, and with what share of its area (in degree space, fine at 10 km)?
BOX_CELLS = []
for jj in range(merra2.native_cell(BBOX[1], BBOX[0])["j"], merra2.native_cell(BBOX[3], BBOX[0])["j"] + 1):
    for ii in range(merra2.native_cell(BBOX[1], BBOX[0])["i"], merra2.native_cell(BBOX[1], BBOX[2])["i"] + 1):
        lat_c, lon_c = -90 + 0.5 * jj, -180 + 0.625 * ii
        ov = (max(0, min(BBOX[3], lat_c + 0.25) - max(BBOX[1], lat_c - 0.25)) * max(0, min(BBOX[2], lon_c + 0.3125) - max(BBOX[0], lon_c - 0.3125)))
        BOX_CELLS.append({"i": ii, "j": jj, "lat_c": lat_c, "lon_c": lon_c,
                          "box_share_pct": round(100 * ov / ((BBOX[3] - BBOX[1]) * (BBOX[2] - BBOX[0])), 1)})
print("native cells overlapped by the box:", BOX_CELLS)

UA = {"User-Agent": "DuneSight-pipeline/0.1 (open-source research)"}
def power_hourly(year_start, year_end, lat=LAT_C, lon=LON_C):
    url = ("https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=U10M,V10M&community=RE"
           f"&longitude={lon}&latitude={lat}&start={year_start}&end={year_end}&format=JSON&time-standard=UTC")
    for attempt in range(5):
        try:
            resp = requests.get(url, headers=UA, timeout=300); resp.raise_for_status()
            p = resp.json()["properties"]["parameter"]
            df = pd.DataFrame({"U10M": p["U10M"], "V10M": p["V10M"]})
            df.index = pd.to_datetime(df.index, format="%Y%m%d%H", utc=True)
            return df.replace(-999.0, np.nan)
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * (attempt + 1))

check_hour = "2020-01-01T06:00:00"
pw = power_hourly("20200101", "20200101").loc[pd.Timestamp(check_hour, tz="UTC")]
img = slv.filterDate(check_hour, ee.Date(check_hour).advance(1, "hour")).first().select(["U10M", "V10M"])
cands = {"EE pixel at native-cell index (used)": (cell["i"], cell["j"]), "EE pixel at centroid coordinates": naive_ij}
for di in (-1, 0, 1):
    for dj in (-1, 0, 1):
        cands.setdefault(f"EE pixel ({cell['i'] + di},{cell['j'] + dj})", (cell["i"] + di, cell["j"] + dj))
print(f"\nhour {check_hour} UTC; NASA POWER at the centroid: U10M={pw.U10M:.2f} V10M={pw.V10M:.2f} m/s")
match = {}
for name, (i, j) in cands.items():
    x, y = merra2.ee_pixel_centre(T, i, j)
    v = img.reduceRegion(ee.Reducer.first(), ee.Geometry.Point([x, y]), crs="EPSG:4326", crsTransform=T).getInfo()
    match[name] = abs(v["U10M"] - pw.U10M) < 0.006 and abs(v["V10M"] - pw.V10M) < 0.006
    print(f"  {name:40s} i={i} j={j}  U10M={v['U10M']:.3f} V10M={v['V10M']:.3f}  matches POWER: {match[name]}")
assert match["EE pixel at native-cell index (used)"], "native-cell pixel does not match POWER; grid assumption wrong"
SAMPLE_XY = merra2.ee_pixel_centre(T, cell["i"], cell["j"])
print(f"\nsampling Earth Engine at {SAMPLE_XY} (EE coordinates), which holds native cell centre {cell['lat_c']} N {cell['lon_c']} E")

In [ ]:
# WHAT THIS DOES: pulls hourly U10M/V10M for 2000-01-01 to 2024-12-31 from Earth Engine for that cell (one request
# per year, in parallel), caches it in pipeline/raw/<site>/, and reports missing or duplicate hours.
WIND_CACHE = RAW / f"merra2_u10v10_hourly_i{cell['i']}_j{cell['j']}.csv"
YEARS = list(range(2000, 2025))

def merra_year(y, xy=SAMPLE_XY):
    c = slv.filterDate(f"{y}-01-01", f"{y + 1}-01-01").select(["U10M", "V10M"])
    for attempt in range(5):
        try:
            rows = c.getRegion(ee.Geometry.Point(list(xy)), scale=None, crs="EPSG:4326", crsTransform=T).getInfo()
            df = pd.DataFrame(rows[1:], columns=rows[0])
            df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)
            return df[["time", "U10M", "V10M"]]
        except Exception:
            if attempt == 4:
                raise
            time.sleep(5 * (attempt + 1))

if WIND_CACHE.exists() and not REFRESH:
    hourly = pd.read_csv(WIND_CACHE, parse_dates=["time"])
    print(f"loaded cached MERRA-2 series: {len(hourly)} rows ({WIND_CACHE.relative_to(config.ROOT)})")
else:
    t0 = time.time()
    with ThreadPoolExecutor(6) as ex:
        hourly = pd.concat(list(ex.map(merra_year, YEARS)), ignore_index=True)
    hourly.to_csv(WIND_CACHE, index=False)
    print(f"queried Earth Engine: {len(hourly)} rows in {time.time() - t0:.0f} s")

hourly["time"] = pd.to_datetime(hourly["time"], utc=True)
dups = int(hourly.time.duplicated().sum())
hourly = hourly.drop_duplicates("time").set_index("time").sort_index()
full = pd.date_range("2000-01-01T00:00Z", "2024-12-31T23:00Z", freq="h")
hourly = hourly.reindex(full)
miss = hourly.U10M.isna() | hourly.V10M.isna()
print(f"expected hours {len(full)}, present {int((~miss).sum())}, missing {int(miss.sum())}, duplicates dropped {dups}")
if miss.any():
    print(hourly[miss].groupby(hourly[miss].index.year).size().rename("missing_hours"))
print("speed (m/s):", (np.hypot(hourly.U10M, hourly.V10M)).describe().round(2).to_dict())

In [ ]:
# WHAT THIS DOES: computes yearly and whole-period DP, RDP, RDD and RDP/DP, the DP class and the 16-sector sand rose
# from the hourly MERRA-2 series.
ok = ~miss
yr_rows, period = wind.fryberger_by_year(hourly.index.year[ok], hourly.U10M[ok].values, hourly.V10M[ok].values)
yearly = pd.DataFrame([{k: r[k] for k in ("year", "DP", "RDP", "RDD_deg", "RDP_DP", "pct_hours_above_vt", "n_valid", "n_missing")}
                       for r in yr_rows])
yearly["dp_class"] = yearly.DP.map(wind.dp_class)
print(yearly.round(2).to_string(index=False))
print(f"\nWhole period 2000-2024 (mean annual): DP = {period['DP']:.1f} VU ({period['dp_class']}), RDP = {period['RDP']:.1f} VU, "
      f"RDD = {period['RDD_deg']:.1f} deg, RDP/DP = {period['RDP_DP']:.3f}")
print(f"yearly DP range {yearly.DP.min():.1f}-{yearly.DP.max():.1f} VU (median {yearly.DP.median():.1f}); "
      f"yearly RDD range {yearly.RDD_deg.min():.1f}-{yearly.RDD_deg.max():.1f} deg; "
      f"hours above 12 kt: {yearly.pct_hours_above_vt.mean():.2f} % on average")
rose = pd.DataFrame({"sector_from_deg": np.arange(16) * 22.5, "dp": period["rose_from"]})
rose["share_pct"] = 100 * rose.dp / rose.dp.sum()
print("\nsand rose (mean annual DP by FROM sector):")
print(rose.round(2).to_string(index=False))
yearly.round(3).to_csv(OUT / "wind_yearly_merra2.csv", index=False)
rose.round(3).to_csv(OUT / "sand_rose_merra2.csv", index=False)

In [ ]:
# WHAT THIS DOES: draws the sand rose (bars point to the FROM direction, length = mean annual DP) with the resultant
# drift arrow (toward RDD), and the yearly DP and RDD series.
fig = plt.figure(figsize=(12, 4.2))
ax = fig.add_subplot(1, 3, 1, projection="polar")
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
ax.bar(np.radians(rose.sector_from_deg), rose.dp, width=np.radians(20), color="#4a7fb5", edgecolor="white", linewidth=1)
ax.annotate("", xy=(np.radians(period["RDD_deg"]), rose.dp.max()), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color="#c0392b", lw=2))
ax.set_title(f"Sand rose (FROM), DP {period['DP']:.0f} VU/yr\nred arrow: RDD {period['RDD_deg']:.0f}° (sand moves to)", fontsize=9)
ax.tick_params(labelsize=7)
ax2 = fig.add_subplot(1, 3, 2)
ax2.bar(yearly.year, yearly.DP, color="#4a7fb5"); ax2.bar(yearly.year, yearly.RDP, color="#9cc3e6", width=0.4)
ax2.set_title("Yearly DP (wide) and RDP (narrow), VU", fontsize=9); ax2.tick_params(labelsize=7)
ax3 = fig.add_subplot(1, 3, 3)
ax3.plot(yearly.year, yearly.RDD_deg, "o-", color="#4a7fb5", lw=2, ms=4)
ax3.set_ylim(0, 360); ax3.set_yticks(range(0, 361, 45)); ax3.set_title("Yearly RDD (deg, direction sand moves to)", fontsize=9)
ax3.tick_params(labelsize=7); ax3.grid(alpha=0.3)
fig.suptitle(f"MERRA-2 hourly 10 m wind, cell centred {cell['lat_c']} N {cell['lon_c']} E, 2000-2024", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "wind_sand_rose_merra2.png", dpi=110); plt.show()

In [ ]:
# WHAT THIS DOES: checks whether the drift direction is physically plausible for the UAE (NW shamal -> sand toward
# the SE) by breaking DP and RDD down by month and by time of day.
h = hourly[ok].copy()
h["kt"] = wind.speed_kt(h.U10M, h.V10M)
h["from"] = wind.from_dir_deg(h.U10M, h.V10M)
def grp_stats(g):
    r = wind.fryberger(g.U10M.values, g.V10M.values)
    # scale so the numbers are this group's contribution to the mean annual DP (the group's hour-mean DP times its
    # share of all hours; the contributions of all months, or all hours of day, add up to the mean annual DP)
    share = len(g) / len(h)
    return pd.Series({"DP_contrib": r["DP"] * share, "RDD_deg": r["RDD_deg"], "RDP_DP": r["RDP_DP"],
                      "pct_hours_above_12kt": r["pct_hours_above_vt"], "mean_kt": g.kt.mean()})
by_month = h.groupby(h.index.month).apply(grp_stats)
by_month.index.name = "month"
print("contribution to mean annual DP by calendar month (VU/yr), with that month's RDD:")
print(by_month.round(2).to_string())
print(f"check: monthly contributions sum to {by_month.DP_contrib.sum():.1f} VU/yr (mean annual DP {period['DP']:.1f}; equal up to leap-year weighting)")
by_hour = h.groupby(h.index.hour).apply(grp_stats)
by_hour.index.name = "hour_utc"
print("\nby hour of day (UTC; local time = UTC+4):")
print(by_hour.round(2).to_string())
strong = h[h.kt > 12]
print(f"\nhours above 12 kt: {len(strong)} ({100 * len(strong) / len(h):.2f} %); FROM-direction quartiles of those hours:",
      np.percentile(strong["from"], [25, 50, 75]).round(0))
print("share of above-threshold hours by FROM sector (%):")
print((strong.groupby(wind.sector_index(strong["from"])).size() / len(strong) * 100).rename(lambda k: k * 22.5).round(1).to_string())

### Cross-check with NASA POWER

NASA POWER's hourly meteorology is derived from MERRA-2 on the same native grid, so this checks **our extraction and processing** (grid cell, time stamps, units, the DP code), not the physics. POWER hourly starts in 2001, so the comparison covers 2001-2024.

In [ ]:
# WHAT THIS DOES: downloads NASA POWER hourly U10M/V10M at the box centroid for 2001-2024 (one request per year,
# cached), compares it hour by hour with the Earth Engine series, and compares yearly DP and RDD.
POWER_CACHE = RAW / "power_u10v10_hourly_centroid.csv"
P_YEARS = list(range(2001, 2025))
if POWER_CACHE.exists() and not REFRESH:
    power = pd.read_csv(POWER_CACHE, index_col=0, parse_dates=True)
    print(f"loaded cached POWER series: {len(power)} rows")
else:
    with ThreadPoolExecutor(4) as ex:
        power = pd.concat(list(ex.map(lambda y: power_hourly(f"{y}0101", f"{y}1231"), P_YEARS)))
    power.to_csv(POWER_CACHE)
    print(f"downloaded POWER: {len(power)} rows")
power.index = pd.to_datetime(power.index, utc=True)
pmiss = power.U10M.isna() | power.V10M.isna()
print(f"POWER hours {len(power)}, missing {int(pmiss.sum())}")

j = hourly.join(power.add_suffix("_power"), how="inner").dropna()
du, dv = j.U10M - j.U10M_power, j.V10M - j.V10M_power
worst = (j.V10M - j.V10M_power).abs().idxmax()
print(f"largest single-hour difference: {worst} UTC; EE U,V = {j.U10M[worst]:.2f}, {j.V10M[worst]:.2f}; POWER U,V = {j.U10M_power[worst]:.2f}, {j.V10M_power[worst]:.2f}")
print(f"hours differing by more than 0.01 m/s (beyond POWER's rounding): U {int((du.abs() > 0.0101).sum())}, V {int((dv.abs() > 0.0101).sum())}")
print(f"hourly comparison over {len(j)} common hours: |dU| mean {du.abs().mean():.4f} max {du.abs().max():.3f} m/s; "
      f"|dV| mean {dv.abs().mean():.4f} max {dv.abs().max():.3f} m/s; r(U) {np.corrcoef(j.U10M, j.U10M_power)[0, 1]:.5f}, "
      f"r(V) {np.corrcoef(j.V10M, j.V10M_power)[0, 1]:.5f}  (POWER rounds to 0.01 m/s)")

pok = ~pmiss
p_rows, p_period = wind.fryberger_by_year(power.index.year[pok], power.U10M[pok].values, power.V10M[pok].values)
cmp = pd.DataFrame([{"year": a["year"], "DP_merra2_ee": a["DP"], "DP_power": b["DP"], "RDD_merra2_ee": a["RDD_deg"],
                     "RDD_power": b["RDD_deg"]} for a, b in zip([r for r in yr_rows if r["year"] >= 2001], p_rows)])
cmp["DP_diff_pct"] = 100 * (cmp.DP_power - cmp.DP_merra2_ee) / cmp.DP_merra2_ee
cmp["RDD_diff_deg"] = wind.angle_diff(cmp.RDD_power, cmp.RDD_merra2_ee)
print(cmp.round(2).to_string(index=False))
ee_2001 = wind.fryberger_by_year(hourly.index.year[ok & (hourly.index.year >= 2001)],
                                 hourly.U10M[ok & (hourly.index.year >= 2001)].values, hourly.V10M[ok & (hourly.index.year >= 2001)].values)[1]
POWER_AGREEMENT = {
    "period": "2001-2024", "common_hours": int(len(j)),
    "DP_merra2_ee": round(ee_2001["DP"], 2), "DP_power": round(p_period["DP"], 2),
    "DP_diff_pct": round(100 * (p_period["DP"] - ee_2001["DP"]) / ee_2001["DP"], 3),
    "RDD_merra2_ee": round(ee_2001["RDD_deg"], 2), "RDD_power": round(p_period["RDD_deg"], 2),
    "RDD_diff_deg": round(float(wind.angle_diff(p_period["RDD_deg"], ee_2001["RDD_deg"])), 3),
    "max_yearly_DP_diff_pct": round(float(cmp.DP_diff_pct.abs().max()), 3),
    "max_yearly_RDD_diff_deg": round(float(cmp.RDD_diff_deg.max()), 3),
}
print("\nwhole-period agreement:", POWER_AGREEMENT)
cmp.round(3).to_csv(OUT / "wind_power_crosscheck.csv", index=False)

In [ ]:
# WHAT THIS DOES: sensitivity check for the grid-cell choice: repeats DP/RDD for 2020-2024 using (a) the pixel Earth
# Engine would sample at the centroid's coordinates, if that is a different native cell, and (b) any other native cell
# the box overlaps, to show how much the cell matters.
sub = hourly[ok & (hourly.index.year >= 2020)]
cr, cper = wind.fryberger_by_year(sub.index.year, sub.U10M.values, sub.V10M.values)
print(f"2020-2024 used cell (native centre {cell['lat_c']}, {cell['lon_c']}): DP {cper['DP']:.1f}, RDD {cper['RDD_deg']:.1f}, RDP/DP {cper['RDP_DP']:.3f}")
alt_cells = []
if tuple(naive_ij) != (cell["i"], cell["j"]):
    alt_cells.append(("naive cell", tuple(naive_ij)))
else:
    print("the pixel Earth Engine samples at the centroid's coordinates holds the same native cell here (no naive-cell error at this site)")
alt_cells += [(f"cell holding {c['box_share_pct']} % of the box", (c["i"], c["j"])) for c in BOX_CELLS
              if (c["i"], c["j"]) != (cell["i"], cell["j"]) and c["box_share_pct"] > 0 and (c["i"], c["j"]) not in [a[1] for a in alt_cells]]
SENSITIVITY = []
for label, (ii, jj) in alt_cells:
    xy = merra2.ee_pixel_centre(T, ii, jj)
    cache = RAW / f"merra2_u10v10_hourly_i{ii}_j{jj}_2020-2024.csv"
    if cache.exists() and not REFRESH:
        nh = pd.read_csv(cache, parse_dates=["time"])
    else:
        with ThreadPoolExecutor(5) as ex:
            nh = pd.concat(list(ex.map(lambda y: merra_year(y, xy), range(2020, 2025))), ignore_index=True)
        nh.to_csv(cache, index=False)
    nh["time"] = pd.to_datetime(nh["time"], utc=True)
    nr, nper = wind.fryberger_by_year(nh.time.dt.year.values, nh.U10M.values, nh.V10M.values)
    nat = (-90 + 0.5 * jj, -180 + 0.625 * ii)
    SENSITIVITY.append({"label": label, "i": ii, "j": jj, "lat_c": nat[0], "lon_c": nat[1], "DP_2020_2024": round(nper["DP"], 1),
                        "RDD_2020_2024": round(nper["RDD_deg"], 1), "RDP_DP_2020_2024": round(nper["RDP_DP"], 3)})
    print(f"2020-2024 {label} (native centre {nat[0]}, {nat[1]}): DP {nper['DP']:.1f}, RDD {nper['RDD_deg']:.1f}, RDP/DP {nper['RDP_DP']:.3f}")


In [ ]:
# WHAT THIS DOES: prints, from this run's numbers, the wind facts the written note below rests on: the cell, DP, RDP,
# RDD, RDP/DP, class, yearly ranges, the main sand-rose sectors, the NW-mode share and the POWER agreement.
top = rose.sort_values("dp", ascending=False).head(4)
nw = rose[rose.sector_from_deg.between(270, 337.5)].share_pct.sum()
south = rose[rose.sector_from_deg.between(135, 225)].share_pct.sum()
L = [f"- MERRA-2 native cell i={cell['i']} j={cell['j']}, centre {cell['lat_c']} N {cell['lon_c']} E, spans {cell['south']}-{cell['north']} N, "
     f"{cell['west']}-{cell['east']} E; box share inside it {next(c['box_share_pct'] for c in BOX_CELLS if (c['i'], c['j']) == (cell['i'], cell['j']))} %; "
     f"missing hours {int(miss.sum())} of {len(full)}",
     f"- DP {period['DP']:.1f} VU/yr (**{period['dp_class']}**), RDP {period['RDP']:.1f} VU/yr, RDD {period['RDD_deg']:.1f} deg (sand moves to), RDP/DP {period['RDP_DP']:.3f}",
     f"- yearly DP {yearly.DP.min():.1f}-{yearly.DP.max():.1f} VU (median {yearly.DP.median():.1f}); yearly RDD {yearly.RDD_deg.min():.1f}-{yearly.RDD_deg.max():.1f} deg; "
     f"hours above 12 kt {yearly.pct_hours_above_vt.mean():.2f} %",
     f"- sand rose: top FROM sectors " + ", ".join(f"{r.sector_from_deg:g} deg ({r.share_pct:.1f} %)" for r in top.itertuples())
     + f"; W-NNW (270-337.5) {nw:.1f} %, SE-SW (135-225) {south:.1f} % of DP",
     f"- month with the largest DP: {int(by_month.DP_contrib.idxmax())} ({by_month.DP_contrib.max():.1f} VU/yr, RDD {by_month.RDD_deg[by_month.DP_contrib.idxmax()]:.0f} deg)",
     f"- NASA POWER 2001-2024: DP differs {POWER_AGREEMENT['DP_diff_pct']} %, RDD {POWER_AGREEMENT['RDD_diff_deg']} deg; "
     f"max yearly {POWER_AGREEMENT['max_yearly_DP_diff_pct']} % / {POWER_AGREEMENT['max_yearly_RDD_diff_deg']} deg",
     ] + [f"- sensitivity, {s['label']} (centre {s['lat_c']} N {s['lon_c']} E), 2020-2024: DP {s['DP_2020_2024']} vs {cper['DP']:.1f}, "
          f"RDD {s['RDD_2020_2024']} vs {cper['RDD_deg']:.1f} deg" for s in SENSITIVITY]
display(Markdown(f"**Wind numbers from this run ({SID}):**\n\n" + "\n".join(L)))


In [ ]:
# WHAT THIS DOES: shows the written interpretation of the wind results for this site (pipeline/site_notes/<site>/01_wind.md).
_f = NOTES / "01_wind.md"
display(Markdown(_f.read_text()) if _f.exists() else Markdown(f"**No written note yet for {SID}** ({_f.relative_to(config.ROOT)})."))


In [ ]:
# WHAT THIS DOES: writes public/data/<site>/wind_summary.json in the handoff §4 format with the real values (plus a
# note on conventions and data handling), then rebuilds public/data/sites.json so this file shows as "real".
out = {
    "location": {"lat": round(LAT_C, 4), "lon": round(LON_C, 4)},
    "period": "2000-2024",
    "source": "NASA MERRA-2 U10M/V10M hourly",
    "threshold_kt": 12,
    "DP": round(period["DP"], 1),
    "RDP": round(period["RDP"], 1),
    "RDD_deg": round(period["RDD_deg"], 1),
    "RDP_DP": round(period["RDP_DP"], 3),
    "dp_class": period["dp_class"],
    "yearly": [{"year": int(r.year), "DP": round(r.DP, 1), "RDP": round(r.RDP, 1), "RDD_deg": round(r.RDD_deg, 1)} for r in yearly.itertuples()],
    "sand_rose": [{"sector_deg": float(r.sector_from_deg), "dp": round(r.dp, 2)} for r in rose.itertuples()],
    "placeholder": False,
    "note": (
        "Fryberger & Dean (1979) drift potential, DP = sum V^2 (V - 12 kt)/100 * t with V the hourly-mean 10 m wind "
        "in knots and t each hour's share of its year in percent; DP, RDP are mean annual values in vector units (VU/yr). "
        "RDD_deg is the direction sand moves TO (resultant of downwind drift vectors), degrees clockwise from north. "
        "sand_rose.sector_deg is the direction the wind blows FROM (16 sectors of 22.5 deg centred on sector_deg); "
        "sand_rose.dp sums to DP. "
        f"Data: MERRA-2 tavg1_2d_slv_Nx via Earth Engine NASA/GSFC/MERRA/slv/2, native cell centred {cell['lat_c']} N "
        f"{cell['lon_c']} E (spans {cell['south']}-{cell['north']} N, {cell['west']}-{cell['east']} E; 0.5 x 0.625 deg, about 55 x 63 km) "
        f"containing the box centroid ({next(c['box_share_pct'] for c in BOX_CELLS if (c['i'], c['j']) == (cell['i'], cell['j']))} % of the box lies in it); Earth Engine's grid is "
        "offset from the native one, so the pixel was chosen by native index and checked against NASA POWER. "
        f"Missing hours: {int(miss.sum())} of {len(full)}. "
        f"NASA POWER cross-check 2001-2024: DP differs by {POWER_AGREEMENT['DP_diff_pct']} %, RDD by {POWER_AGREEMENT['RDD_diff_deg']} deg "
        "(same underlying MERRA-2, so this checks processing only). "
        "Caveat: a ~50 km reanalysis of hourly means damps gusts, so absolute DP is likely biased low against station "
        "anemometers and dp_class is approximate; RDD and RDP_DP are the more robust outputs. "
        "Computed by pipeline/01_scenes_and_wind.ipynb."
    ),
}
(PUB / "wind_summary.json").write_text(json.dumps(out, indent=1, allow_nan=False) + "\n")
idx = site_index.write_index()
me = next(s for s in idx["sites"] if s["id"] == SID)
print(f"wrote {(PUB / 'wind_summary.json').relative_to(config.ROOT)}")
print(f"public/data/sites.json: {SID} status={me['status']}; files: {me['files']}")
print(json.dumps({k: v for k, v in out.items() if k not in ('yearly', 'sand_rose', 'note')}, indent=1))